## Importações

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pickle
import os

# modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]
years = range(2019, 2025)

portfolios = {}

k_values = [1, 10]

for year in years:
    for centrality in centralities:
        for k in k_values:
            cols = pd.read_csv(f"../../data/06_portfolios/{centrality}_{year}_{k}.csv", index_col="Date").columns.tolist()
            try:
                portfolios[f"{centrality}_{year}_{k}"] = pd.read_parquet(f"../../data/02_clean/returns_{year-9}_{year}.parquet")[cols]
            except:
                portfolios[f"{centrality}_{year}_{k}"] = pd.read_parquet(f"../../data/02_clean/returns_{year-k+1}_{year}.parquet")[cols]

# for year in years:
#     for centrality in centralities:
#         for n in range(10,110,10):
#             cols = pd.read_csv(f"../../data/06_portfolios/{n}_{centrality}_{year}.csv", index_col="Date").columns.tolist()
#             portfolios[f"{n}_{centrality}_{year}"] = pd.read_parquet(f"../../data/02_clean/returns_{year-9}_{year}.parquet")[cols]

In [54]:
weekly_returns = {}
weekly2_returns = {}
monthly_returns = {}
daily_returns = {}

for name, portfolio in portfolios.items():
    log_returns = np.log1p(portfolio)  # log(1 + r)
    weekly_returns[name] = log_returns.resample("W-FRI").sum().mean(axis=1)
    daily_returns[name] = log_returns.mean(axis=1)
    weekly2_returns[name] = log_returns.resample("2W-FRI").sum().mean(axis=1)
    monthly_returns[name] = log_returns.resample("ME").sum().mean(axis=1)

## Lead-lag

In [55]:
def autocorrelation_matrix(X, lag):
    X_t = X.iloc[lag:]
    X_tk = X.shift(lag).iloc[lag:]

    mu = X_t.mean().values

    Xc_t = X_t.values - mu
    Xc_tk = X_tk.values - mu

    Sigma_k = (Xc_tk.T @ Xc_t) / len(Xc_t)

    var = X.var(ddof=0).values
    D_inv_sqrt = np.diag(1 / np.sqrt(var))

    return D_inv_sqrt @ Sigma_k @ D_inv_sqrt


def plot_antisymmetric_autocorr(
    returns_list,
    column_names,
    labels,
    lags=(1, 2, 3, 4),
    figsize=(12, 8),
    cmap="Blues",
    title_prefix="Y",
    annot=False,
    diff=True,
    fig_title=None
):
    X = pd.concat(returns_list, axis=1)
    X.columns = column_names

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.flatten()

    if len(lags) == 1:
        axes = [axes]

    for ax, lag in zip(axes, lags):
        T = autocorrelation_matrix(X, lag)

        if diff:
            A = pd.DataFrame(
                T - T.T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag}) - {title_prefix}'({lag})"
        else:
            A = pd.DataFrame(
                T,
                index=labels,
                columns=labels
            )
            title = f"{title_prefix}({lag})"

        sns.heatmap(A, ax=ax, cmap=cmap, center=0, annot=annot)
        ax.set_title(title)

    if fig_title:
        fig.suptitle(fig_title, fontsize=14)
    plt.tight_layout()
    plt.show()

### Central Peripheral

In [56]:
years = range(2019, 2025)

for year in years:
    for k in k_values:
        try:
            R1 = daily_returns[f"peripheral_{year}_{k}"]
            R2 = daily_returns[f"central_{year}_{k}"]
            R3 = weekly_returns[f"peripheral_{year}_{k}"]
            R4 = weekly_returns[f"central_{year}_{k}"]
            R5 = weekly2_returns[f"peripheral_{year}_{k}"]
            R6 = weekly2_returns[f"central_{year}_{k}"]
            R7 = monthly_returns[f"peripheral_{year}_{k}"]
            R8 = monthly_returns[f"central_{year}_{k}"]

            # plot_antisymmetric_autocorr(
            #     returns_list=[R1, R2],
            #     column_names=[
            #         "Peripheral", "Central"
            #     ],
            #     labels=["P", "C"],
            #     annot=True,
            #     fig_title=f"{year}",
            #     lags=(1, 2),
            #     figsize=(8,4)
            # )

        except KeyError as e:
            print(f"Missing data for {year}: {e}")

        # =============================
        # 5. Lead–lag matrix
        # =============================
        matrix = []

        # X = pd.concat([R1, R2], axis=1)
        # X.columns = ["Peripheral", "Central"]
        lag_list = [1]

        for lag in lag_list:
            X = pd.concat([R1, R2], axis=1)
            X.columns = ["Peripheral", "Central"]
            acm = autocorrelation_matrix(X, lag)
            lag_matrix = acm - acm.T

            matrix.append(
                lag_matrix[1, 0]
            )

            X = pd.concat([R3, R4], axis=1)
            X.columns = ["Peripheral", "Central"]
            acm = autocorrelation_matrix(X, lag)
            lag_matrix = acm - acm.T

            matrix.append(
                lag_matrix[1, 0]
            )

            X = pd.concat([R5, R6], axis=1)
            X.columns = ["Peripheral", "Central"]
            acm = autocorrelation_matrix(X, lag)
            lag_matrix = acm - acm.T

            matrix.append(
                lag_matrix[1, 0]
            )

            X = pd.concat([R7, R8], axis=1)
            X.columns = ["Peripheral", "Central"]
            acm = autocorrelation_matrix(X, lag)
            lag_matrix = acm - acm.T

            matrix.append(
                lag_matrix[1, 0]
            )

        # =============================
        # 6. DataFrame final
        # =============================
        cols = ["cp1d", "cp1w",
                    "cp2w", "cp1m"]
        # cols = [f"{leadlag}{i}" for i in lag_list for leadlag in leadlags]

        leadlag_df = pd.DataFrame(
            [np.hstack(matrix)],
            columns=cols,
            index=[year]
        )

        leadlag_df.to_csv(
            f"../../data/08_lead_lag/leadlag_df_{year}_{k}.csv"
        )

### Lead-lag considerando Market Cap

In [57]:
import pandas as pd
import numpy as np

In [58]:
df_mcap = pd.read_csv(
    "../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv"
)

In [63]:
import pandas as pd
import numpy as np

# Assuming autocorrelation_matrix function is defined elsewhere
# from your_module import autocorrelation_matrix 

years = range(2019, 2025)

# Load metadata and returns once outside the loop for efficiency
df_mcap_meta = pd.read_csv("../../data/07_portfolios_metadata/all_tickers_complete_metadata.csv")
df_ret_full = pd.read_csv("../../data/02_clean/returns_clean_401.csv").set_index("Date")
df_ret_full.index = pd.to_datetime(df_ret_full.index)
df_ret_full = df_ret_full[~df_ret_full.index.duplicated(keep='first')].sort_index()
df_ret_full = df_ret_full[df_ret_full.index >= pd.Timestamp(2019, 1, 1)]

for year in years:
    try:
        # 1. Load returns for the calculation window (e.g., year-9 to year)
        returns_prev = pd.read_parquet(f"../../data/02_clean/returns_{year-9}_{year}.parquet")
        
        # 2. Market cap cleaning and quantile logic
        df_mcap_year = df_mcap_meta[(df_mcap_meta["Ticker"].isin(returns_prev.columns)) &
        (df_mcap_meta["Ticker"].isin(df_ret_full.columns))].copy()
        mcap_col = f"mcap_{year}"

        # Clean numeric data (handle dots and commas)
        df_mcap_year[mcap_col] = (
            df_mcap_year[mcap_col]
            .replace("#ERROR!", np.nan)
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df_mcap_year[mcap_col] = pd.to_numeric(df_mcap_year[mcap_col], errors="coerce")
        df_mcap_year = df_mcap_year.dropna(subset=[mcap_col])

        # Define Portfolios based on p80/p20 quantiles
        p80 = df_mcap_year[mcap_col].quantile(0.8)
        p20 = df_mcap_year[mcap_col].quantile(0.2)

        large_tickers = df_mcap_year[df_mcap_year[mcap_col] >= p80]["Ticker"].tolist()
        small_tickers = df_mcap_year[df_mcap_year[mcap_col] <= p20]["Ticker"].tolist()

        # 3. Calculate Portfolio Returns (Log returns)
        # R1/R2: Daily
        log_rets = np.log1p(returns_prev)
        R1 = log_rets[large_tickers].mean(axis=1)
        R2 = log_rets[small_tickers].mean(axis=1)

        # R3/R4: Weekly (Friday)
        R3 = log_rets[large_tickers].resample("W-FRI").sum().mean(axis=1)
        R4 = log_rets[small_tickers].resample("W-FRI").sum().mean(axis=1)

        # R5/R6: 2-Week
        R5 = log_rets[large_tickers].resample("2W-FRI").sum().mean(axis=1)
        R6 = log_rets[small_tickers].resample("2W-FRI").sum().mean(axis=1)

        # R7/R8: Monthly
        R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
        R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)

        # 4. Lead-Lag Matrix Calculation
        matrix = []
        lag = 1
        
        # Pairs to iterate through (Small vs Large at different frequencies)
        pairs = [(R1, R2), (R3, R4), (R5, R6), (R7, R8)]
        
        for s_ret, l_ret in pairs:
            X = pd.concat([s_ret, l_ret], axis=1).dropna()
            X.columns = ["Large", "Small"]
            
            acm = autocorrelation_matrix(X, lag)
            # Cross-autocorrelation asymmetry: (Large leads Small) - (Small leads Large)
            lag_matrix = acm - acm.T
            matrix.append(lag_matrix[1, 0])

        # 5. Save Results
        cols = ["ls1d", "ls1w", "ls2w", "ls1m"]
        leadlag_df = pd.DataFrame([matrix], columns=cols, index=[year])
        leadlag_df.to_csv(f"../../data/08_lead_lag/marketcap_leadlag_df_{year}.csv")
        
    except Exception as e:
        print(f"Error processing year {year}: {e}")
        continue

C:\Users\groque\AppData\Local\Temp\ipykernel_25388\3211324963.py:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
C:\Users\groque\AppData\Local\Temp\ipykernel_25388\3211324963.py:60: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
C:\Users\groque\AppData\Local\Temp\ipykernel_25388\3211324963.py:59: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R7 = log_rets[large_tickers].resample("M").sum().mean(axis=1)
C:\Users\groque\AppData\Local\Temp\ipykernel_25388\3211324963.py:60: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  R8 = log_rets[small_tickers].resample("M").sum().mean(axis=1)
C:\Users\groque\AppData\Local\Temp\ipykernel_25388\3211324963.py:59: Fut